# P05: data cleaning + RAG (Person 2)

AI-assisted code (mark this in the final notebook). Builds on Person 1's `data.py` and `config.py`, and on sir's RAG_Pipeline_Lab (embeddings, FAISS, BM25, RRF).

**Run in order.** Part A cleans the data and saves ID files. Part B builds the RAG index and compares retrieval settings on the validation set.

Share these files with the group when done: `clean_train_ids.json`, `clean_val_ids.json`, `clean_test_ids.json`, `excluded_ids.json`, `filter_report.json`.

## Part A: clean the data

### A1. Setup

In [ ]:
import json, re, random, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from data import load_data, run_sql, same_result
from config import CONFIG, SEED, QUICK_RUN, set_seed
set_seed()

TEST_SIZE = 200      # decide after you see the clean counts below
VAL_SIZE = 100
QUICK_TEST = 20      # first 20 test ids = QUICK_RUN subset
AUDIT_N = 60         # test examples to review by hand
CACHE = Path("clean_data"); CACHE.mkdir(exist_ok=True)   # do not commit this folder

### A2. Load the raw data

In [ ]:
ds = load_data()
raw_train, raw_test = ds["train"], ds["test"]
print(len(raw_train), len(raw_test))

### A3. Check every gold query
A gold query is kept only if it is a SELECT, runs without error, and gives a non-trivial result.

In [ ]:
def check_example(ex):
    """Return a reason string: 'ok' or why the example must be removed."""
    sql = ex["sql"].strip()
    if not sql.lower().startswith(("select", "with")):
        return "not_select"
    rows, err = run_sql(ex["sql_context"], sql)
    if err is not None:
        e = err.lower()
        if "no such table" in e:    return "missing_table"
        if "no such column" in e:   return "missing_column"
        if "no such function" in e: return "unsupported_function"
        if "syntax error" in e:     return "syntax_dialect"
        if "timed out" in e:        return "timeout"
        if "setup error" in e:      return "bad_context"
        return "other_error"
    if not rows:
        return "empty_result"
    if len(rows) == 1 and len(rows[0]) == 1 and rows[0][0] in (0, None, ""):
        return "trivial_result"
    return "ok"

def run_checks(split, name):
    cache = CACHE / f"checks_{name}.json"
    if cache.exists():
        return json.load(open(cache))
    res = {}
    for ex in tqdm(split, desc=name):
        res[str(ex["id"])] = check_example(ex)
    json.dump(res, open(cache, "w"))
    return res

test_checks = run_checks(raw_test, "test")      # about a minute
train_checks = run_checks(raw_train, "train")   # a few minutes the first time, cached after

def report(res, name):
    c = Counter(res.values())
    df = pd.DataFrame({"count": c}).sort_values("count", ascending=False)
    df["pct"] = (df["count"] / len(res) * 100).round(1)
    print(f"\n{name}: {len(res)} checked")
    display(df)

report(test_checks, "official test")
report(train_checks, "official train")

### A4. Remove duplicates and leakage, then split
Clean first, then split into train, validation and test. The test set is spread evenly across `sql_complexity` levels.

In [ ]:
norm = lambda s: re.sub(r"\s+", " ", s.lower().strip())

train_idx = {i: k for k, i in enumerate(raw_train["id"])}
test_idx = {i: k for k, i in enumerate(raw_test["id"])}
train_prompt = dict(zip(raw_train["id"], map(norm, raw_train["sql_prompt"])))
test_prompt = dict(zip(raw_test["id"], map(norm, raw_test["sql_prompt"])))
test_cx = dict(zip(raw_test["id"], raw_test["sql_complexity"]))
all_test_prompts = set(test_prompt.values())

# train: keep clean examples, drop duplicates and anything whose question appears in test
seen, clean_train, dropped = set(), [], Counter()
for sid, reason in train_checks.items():
    if reason != "ok":
        continue
    i = int(sid); p = train_prompt[i]
    if p in all_test_prompts:
        dropped["question_in_test"] += 1; continue
    if p in seen:
        dropped["duplicate_question"] += 1; continue
    seen.add(p); clean_train.append(i)
print("clean train after dedupe:", len(clean_train), dict(dropped))

# validation: carved out of the clean train pool
rng = random.Random(SEED)
val_ids = rng.sample(clean_train, VAL_SIZE)
val_set = set(val_ids)
train_ids = [i for i in clean_train if i not in val_set]

# test candidates: clean examples, equal share per complexity level
def stratified_sample(ids, groups, n, seed=SEED):
    rng = random.Random(seed)
    by = {}
    for i, g in zip(ids, groups):
        by.setdefault(g, []).append(i)
    for v in by.values():
        rng.shuffle(v)
    out = []
    while len(out) < n and any(by.values()):
        for g in sorted(by):
            if by[g] and len(out) < n:
                out.append(by[g].pop())
    rng.shuffle(out)
    return out

clean_test = [int(s) for s, r in test_checks.items() if r == "ok"]
print("clean test pool:", len(clean_test))
cand = stratified_sample(clean_test, [test_cx[i] for i in clean_test], TEST_SIZE + 40)
print("test candidates:", len(cand), "| train:", len(train_ids), "| val:", len(val_ids))

### A5. Manual audit (do this by hand)
Some examples pass every filter and are still bad: the gold SQL does not answer the question, the question is ambiguous, or the answer is trivial (for example the whole table). Read the examples below. Split the work: each person reads about 10.

In [ ]:
def show_audit(split, index, ids, start=0, n=10):
    for i in ids[start:start + n]:
        ex = split[index[i]]
        rows, err = run_sql(ex["sql_context"], ex["sql"])
        print("=" * 70)
        print(f"id {i} | {ex['sql_complexity']} | {ex['domain']}")
        print("Q:     ", ex["sql_prompt"])
        print("SCHEMA:", ex["sql_context"][:350].replace("\n", " "))
        print("GOLD:  ", ex["sql"])
        print("RESULT:", (rows or [])[:5], f"({len(rows or [])} rows)")

AUDIT_START = 0     # change to 10, 20, ... up to AUDIT_N
show_audit(raw_test, test_idx, cand[:AUDIT_N], start=AUDIT_START, n=10)

Write down flagged ids with a reason. Example: `{4323: "gold SQL ignores the BRICS filter"}`. Optionally review about 20 train examples the same way with `show_audit(raw_train, train_idx, train_ids, 0, 10)`.

In [ ]:
FLAGGED_TEST = {}    # id: reason  (only ids from the audited candidates)
FLAGGED_TRAIN = {}   # id: reason

test_ids = [i for i in cand if i not in FLAGGED_TEST][:TEST_SIZE]
train_ids = [i for i in train_ids if i not in FLAGGED_TRAIN]
print("manual audit flag rate (test):", f"{len(FLAGGED_TEST)}/{AUDIT_N}")
print("final sizes -> train:", len(train_ids), "| val:", len(val_ids), "| test:", len(test_ids))

### A6. Overlap checks and save

In [ ]:
train_p = {train_prompt[i] for i in train_ids}
val_p = {train_prompt[i] for i in val_ids}
test_p = {test_prompt[i] for i in test_ids}
assert not (train_p & val_p) and not (train_p & test_p) and not (val_p & test_p), "question overlap!"
print("question overlap between train / val / test: 0")

def pick(split, index, ids):
    return split.select([index[i] for i in ids])

train_pool = pick(raw_train, train_idx, train_ids)
val_ds = pick(raw_train, train_idx, val_ids)
test_ds = pick(raw_test, test_idx, test_ids)

shared_ctx = set(train_pool["sql_context"]) & set(test_ds["sql_context"])
print("test examples whose exact schema also appears in train:",
      sum(c in shared_ctx for c in test_ds["sql_context"]), "of", len(test_ds))

print(pd.Series(test_ds["sql_complexity"]).value_counts())

json.dump(train_ids, open("clean_train_ids.json", "w"))
json.dump(val_ids, open("clean_val_ids.json", "w"))
json.dump(test_ids, open("clean_test_ids.json", "w"))
json.dump({"test": FLAGGED_TEST, "train": FLAGGED_TRAIN}, open("excluded_ids.json", "w"), indent=2)
json.dump({
    "test_checks": dict(Counter(test_checks.values())),
    "train_checks": dict(Counter(train_checks.values())),
    "train_dropped_after_checks": dict(dropped),
    "manual_audit": {"checked": AUDIT_N, "flagged": len(FLAGGED_TEST)},
    "final_sizes": {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)},
}, open("filter_report.json", "w"), indent=2)
print("saved ID files and filter_report.json")

## Part B: RAG

### B1. Settings

In [ ]:
SCHEMA_MODE = "full"          # "full" = whole sql_context, "create" = CREATE statements only (agree this with Person 4)
K = 3
RETRIEVAL = "dense"           # dense | bm25 | hybrid | masked
INDEX_N = 10000 if QUICK_RUN else None    # None = use every clean train example

### B2. Schema helpers and question masking (your own idea)
Masking replaces table names, column names, quoted values and numbers in the question with placeholders, so retrieval matches the *shape* of the question and not its topic words.

In [ ]:
import re

def schema_text(ctx, mode=None):
    mode = mode or SCHEMA_MODE
    if mode == "create":
        stmts = re.findall(r"CREATE\s+TABLE[^;]*;", ctx, flags=re.I | re.S)
        return "\n".join(stmts) if stmts else ctx
    return ctx

def schema_names(ctx):
    """Table and column names from the CREATE TABLE statements."""
    tables, cols = set(), set()
    pat = r"CREATE\s+TABLE\s+(?:IF NOT EXISTS\s+)?[`\"\[]?(\w+)[`\"\]]?\s*\((.*?)\)\s*;"
    for m in re.finditer(pat, ctx, flags=re.I | re.S):
        tables.add(m.group(1))
        depth, cur, parts = 0, "", []
        for ch in m.group(2):
            if ch == "(":
                depth += 1
            if ch == ")":
                depth -= 1
            if ch == "," and depth == 0:
                parts.append(cur); cur = ""
            else:
                cur += ch
        parts.append(cur)
        for p in parts:
            tok = p.strip().split()
            if tok and tok[0].upper() not in ("PRIMARY", "FOREIGN", "UNIQUE", "CONSTRAINT", "CHECK"):
                cols.add(tok[0].strip('`"[]'))
    return tables, cols

def mask_question(q, ctx):
    tables, cols = schema_names(ctx)
    out = q
    names = [(n, "<T>") for n in sorted(tables, key=len, reverse=True)] + \
            [(n, "<C>") for n in sorted(cols, key=len, reverse=True)]
    for name, tag in names:
        for v in sorted({name, name.replace("_", " ")}, key=len, reverse=True):
            out = re.sub(rf"\b{re.escape(v)}s?\b", tag, out, flags=re.I)
    out = re.sub(r"'[^']*'|\"[^\"]*\"", "<V>", out)
    out = re.sub(r"\b\d[\d,\.]*\b", "<N>", out)
    return out

# quick look
for ex in test_ds.select(range(3)):
    print(ex["sql_prompt"], "\n  ->", mask_question(ex["sql_prompt"], ex["sql_context"]))

### B3. Build the indexes (from the clean train pool only)

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(CONFIG["embedder"], device=device)

pool = train_pool if INDEX_N is None else train_pool.shuffle(seed=SEED).select(range(min(INDEX_N, len(train_pool))))
pool_q, pool_ctx, pool_sql = pool["sql_prompt"], pool["sql_context"], pool["sql"]
pool_masked = [mask_question(q, c) for q, c in zip(pool_q, pool_ctx)]

def embed(texts, bar=False):
    return embedder.encode(texts, batch_size=128, normalize_embeddings=True,
                           show_progress_bar=bar).astype("float32")

def build_faiss(vecs):
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index

index_dense = build_faiss(embed(pool_q, bar=True))
index_masked = build_faiss(embed(pool_masked, bar=True))
tokenize = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tokenize(q) for q in pool_q])
print("index size:", len(pool_q))

# leakage check: no validation or test question may be inside the index
pool_norm = {norm(q) for q in pool_q}
leak = pool_norm & (test_p | val_p)
print("val/test questions found in the index:", len(leak))
assert not leak

### B4. `retrieve`

In [ ]:
def _dense(index, text, n):
    _, I = index.search(embed([text]), n)
    return [int(i) for i in I[0] if i >= 0]

def _bm25(text, n):
    return [int(i) for i in np.argsort(-bm25.get_scores(tokenize(text)))[:n]]

def retrieve(question, k=K, mode=RETRIEVAL, schema=None):
    """Return k similar solved examples. mode: dense | bm25 | hybrid (RRF) | masked."""
    if k == 0:
        return []
    if mode == "dense":
        ids = _dense(index_dense, question, k)
    elif mode == "bm25":
        ids = _bm25(question, k)
    elif mode == "hybrid":
        score = {}
        for ranking in (_dense(index_dense, question, 50), _bm25(question, 50)):
            for rank, i in enumerate(ranking):
                score[i] = score.get(i, 0) + 1 / (60 + rank + 1)
        ids = sorted(score, key=score.get, reverse=True)[:k]
    elif mode == "masked":
        ids = _dense(index_masked, mask_question(question, schema or ""), k)
    else:
        raise ValueError(mode)
    return [{"sql_prompt": pool_q[i], "sql_context": pool_ctx[i], "sql": pool_sql[i]} for i in ids]

ex = val_ds[0]
for r in retrieve(ex["sql_prompt"], 3, "masked", ex["sql_context"]):
    print(r["sql_prompt"], "->", r["sql"])

### B5. `generate` (zero-shot and few-shot)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm"], torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()

SYSTEM = "You are a SQLite expert. Write exactly one SQLite SELECT query that answers the question. Return only the SQL."

def build_messages(question, schema, examples=None):
    parts = [f"Schema:\n{schema_text(e['sql_context'])}\nQuestion: {e['sql_prompt']}\nSQL: {e['sql']}"
             for e in (examples or [])]
    parts.append(f"Schema:\n{schema_text(schema)}\nQuestion: {question}\nSQL:")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n\n".join(parts)}]

def clean_sql(text):
    text = re.sub(r"```(?:sql)?", "", text, flags=re.I).strip()
    stmt = text.split(";")[0].strip()
    return stmt + ";" if stmt else ""

def generate(question, schema, examples=None):
    """examples=None -> zero-shot, otherwise few-shot. Returns one SQL string."""
    text = llm_tok.apply_chat_template(build_messages(question, schema, examples),
                                       tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=256, do_sample=False,
                           pad_token_id=llm_tok.pad_token_id or llm_tok.eos_token_id)
    return clean_sql(llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

def zero_shot(q, ctx):
    return generate(q, ctx)

def few_shot(q, ctx, k=K, mode=RETRIEVAL):
    return generate(q, ctx, retrieve(q, k, mode, schema=ctx))

ex = val_ds[0]
print("gold:", ex["sql"])
print("zero:", zero_shot(ex["sql_prompt"], ex["sql_context"]))
print("few: ", few_shot(ex["sql_prompt"], ex["sql_context"]))

### B6. Evaluate (saves per-example results, so you can find where retrieval hurt)

In [ ]:
def evaluate(system_fn, data, name):
    rows = []
    for ex in tqdm(data, desc=name):
        t0 = time.time()
        pred = system_fn(ex["sql_prompt"], ex["sql_context"])
        p_rows, err = run_sql(ex["sql_context"], pred)
        g_rows, _ = run_sql(ex["sql_context"], ex["sql"])
        rows.append({"id": ex["id"], "complexity": ex["sql_complexity"], "question": ex["sql_prompt"],
                     "pred": pred, "gold": ex["sql"], "error": err,
                     "correct": bool(err is None and same_result(p_rows, g_rows)),
                     "sec": round(time.time() - t0, 2)})
    df = pd.DataFrame(rows)
    Path("results").mkdir(exist_ok=True)
    df.to_json(f"results/{name}.json", orient="records", indent=1)
    return df

### B7. Compare retrieval settings on the validation set
Choose settings here. Do not tune on the test set.

In [ ]:
N_EVAL = 20 if QUICK_RUN else len(val_ds)
val_eval = val_ds.select(range(N_EVAL))

runs = {"zero_shot": zero_shot}
for mode in ["dense", "bm25", "hybrid", "masked"]:
    for k in [3, 5]:
        runs[f"{mode}_k{k}"] = (lambda q, c, m=mode, kk=k: few_shot(q, c, kk, m))

table = []
for name, fn in runs.items():
    df = evaluate(fn, val_eval, name)
    table.append({"system": name, "exec_acc": round(df.correct.mean(), 3), "avg_sec": round(df.sec.mean(), 2)})
pd.DataFrame(table)

### B8. Where did retrieval hurt?
Cases that zero-shot got right and retrieval got wrong. Read 3 to 5 and note why (misleading example, copied table names, and so on).

In [ ]:
BEST = "masked_k3"     # change to the best row from B7
a = pd.read_json("results/zero_shot.json")
b = pd.read_json(f"results/{BEST}.json")
m = a.merge(b, on="id", suffixes=("_zero", "_rag"))
hurt = m[m.correct_zero & ~m.correct_rag]
print(len(hurt), "examples where retrieval hurt")
for _, r in hurt.head(5).iterrows():
    print("Q:", r.question_zero, "\n gold:", r.gold_zero, "\n zero:", r.pred_zero, "\n rag: ", r.pred_rag, "\n")

### B9. Final test run (once, at the end)

In [ ]:
RUN_TEST = False           # set True once, with the best setting from B7
BEST_MODE, BEST_K = "masked", 3

if RUN_TEST:
    test_eval = test_ds.select(range(QUICK_TEST)) if QUICK_RUN else test_ds
    res = {"zero_shot": evaluate(zero_shot, test_eval, "test_zero_shot"),
           f"{BEST_MODE}_k{BEST_K}": evaluate(lambda q, c: few_shot(q, c, BEST_K, BEST_MODE), test_eval, "test_best_rag")}
    display(pd.DataFrame([{"system": n, "exec_acc": round(d.correct.mean(), 3),
                           "avg_sec": round(d.sec.mean(), 2)} for n, d in res.items()]))